In [1]:
import gc
import itertools
import os
import random
import sys
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Any, Literal, TypeAlias

import circuitsvis as cv
import einops
import pandas as pd
import plotly.express as px
import requests
import torch as t
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import HTML, IFrame, display
from jaxtyping import Float, Int
from openai import OpenAI
from rich import print as rprint
from rich.table import Table
from sae_lens import (
    SAE,
    ActivationsStore,
    GatedTrainingSAEConfig,
    HookedSAETransformer,
    LanguageModelSAERunnerConfig,
    LoggingConfig,
)
from sae_lens.loading.pretrained_saes_directory import get_pretrained_saes_directory
from sae_vis import SaeVisConfig, SaeVisData, SaeVisLayoutConfig
from tabulate import tabulate
from torch import Tensor
from tqdm.auto import tqdm
from transformer_lens import ActivationCache
from transformer_lens.hook_points import HookPoint
from transformer_lens.utils import get_act_name, test_prompt


def find_arena_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "ARENA_3.0").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find an ARENA_3.0 folder above {start}")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part33_interp_with_saes"

root_dir = find_arena_root(Path.cwd())
exercises_dir = root_dir / "ARENA_3.0" / chapter / "exercises"
section_dir = exercises_dir / section

if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))


def _get_hook_layer(sae: SAE) -> int:
    """Extract the layer number from an SAE's hook name (e.g. 'blocks.7.hook_resid_pre' → 7)."""
    return int(sae.cfg.metadata.hook_name.split(".")[1])


os.chdir(section_dir)

load_dotenv()
# Saves computation time, since we don't need it for the contents of this notebook
t.set_grad_enabled(False)

if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

MAIN = __name__ == "__main__"

import part33_interp_with_saes.tests as tests
import part33_interp_with_saes.utils as utils

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

In [2]:
import part33_interp_with_saes.utils as utils

# Profile memory usage, and delete gemma models if we've loaded them in
namespace = globals().copy() | locals()
utils.profile_pytorch_memory(namespace=namespace, filter_device="mps")

┌────────┬──────────┬──────────┬─────────────┐
│ Name   │ Object   │ Device   │ Size (GB)   │
├────────┼──────────┼──────────┼─────────────┤
└────────┴──────────┴──────────┴─────────────┘


In [3]:
get_pretrained_saes_directory()

{'gemma-scope-2-4b-pt-transcoders-all': PretrainedSAELookup(release='gemma-scope-2-4b-pt-transcoders-all', repo_id='google/gemma-scope-2-4b-pt', model='google/gemma-3-4b-pt', conversion_func='gemma_3', saes_map={'layer_0_width_16k_l0_big': 'transcoder_all/layer_0_width_16k_l0_big', 'layer_0_width_16k_l0_big_affine': 'transcoder_all/layer_0_width_16k_l0_big_affine', 'layer_0_width_16k_l0_small': 'transcoder_all/layer_0_width_16k_l0_small', 'layer_0_width_16k_l0_small_affine': 'transcoder_all/layer_0_width_16k_l0_small_affine', 'layer_0_width_262k_l0_big': 'transcoder_all/layer_0_width_262k_l0_big', 'layer_0_width_262k_l0_big_affine': 'transcoder_all/layer_0_width_262k_l0_big_affine', 'layer_0_width_262k_l0_small': 'transcoder_all/layer_0_width_262k_l0_small', 'layer_0_width_262k_l0_small_affine': 'transcoder_all/layer_0_width_262k_l0_small_affine', 'layer_10_width_16k_l0_big': 'transcoder_all/layer_10_width_16k_l0_big', 'layer_10_width_16k_l0_big_affine': 'transcoder_all/layer_10_width_

In [4]:
metadata_rows = [
    [data.model, data.release, data.repo_id, len(data.saes_map)] for data in get_pretrained_saes_directory().values()
]

# Print all SAE releases, sorted by base model
print(
    tabulate(
        sorted(metadata_rows, key=lambda x: x[0]),
        headers=["model", "release", "repo_id", "n_saes"],
        tablefmt="simple_outline",
    )
)

┌──────────────────────────────────────────┬─────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┬──────────┐
│ model                                    │ release                                             │ repo_id                                                   │   n_saes │
├──────────────────────────────────────────┼─────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────┼──────────┤
│ Qwen/Qwen2.5-7B-Instruct                 │ qwen2.5-7b-instruct-andyrdt                         │ andyrdt/saes-qwen2.5-7b-instruct                          │        7 │
│ Qwen/Qwen3-1.7B-Base                     │ qwen-scope-3-1.7b-base-w32k-l50                     │ Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50                   │       28 │
│ Qwen/Qwen3-1.7B-Base                     │ qwen-scope-3-1.7b-base-w32k-l100                    │ Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_100           

In [5]:
t.set_grad_enabled(False)

gpt2: HookedSAETransformer = HookedSAETransformer.from_pretrained("gpt2-small", device=device)

gpt2_sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    device=str(device),
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


/Users/shubmeister/miniconda3/envs/arena/lib/python3.11/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


In [6]:
print(
    tabulate(
        list(gpt2_sae.cfg.__dict__.items()) + list(gpt2_sae.cfg.metadata.items()),
        headers=["name", "value"],
        tablefmt="simple_outline",
    )
)

┌──────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ name                         │ value                                                                                                                                                                                                                                                                                                                                                                      │
├──────────────────────────────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [7]:
def display_dashboard(
    sae_release="gpt2-small-res-jb",
    sae_id="blocks.7.hook_resid_pre",
    latent_idx=0,
    width=800,
    height=600,
):
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    url = f"https://neuronpedia.org/{neuronpedia_id}/{latent_idx}?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300"

    print(url)
    display(IFrame(url, width=width, height=height))


latent_idx = random.randint(0, gpt2_sae.cfg.d_sae)
display_dashboard(latent_idx=latent_idx)

https://neuronpedia.org/gpt2-small/7-res-jb/21080?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


In [8]:
prompt = "Mitigating the risk of extinction from AI should be a global"
answer = " priority"

# First see how the model does without SAEs
test_prompt(prompt, answer, gpt2)

# Test our prompt, to see what the model says
with gpt2.saes(saes=[gpt2_sae]):
    test_prompt(prompt, answer, gpt2)

# Same thing, done in a different way
gpt2.add_sae(gpt2_sae)
test_prompt(prompt, answer, gpt2)
gpt2.reset_saes()  # Remember to always do this!

# Using `run_with_saes` method in place of standard forward pass
logits = gpt2(prompt, return_type="logits")
logits_sae = gpt2.run_with_saes(prompt, saes=[gpt2_sae], return_type="logits")
answer_token_id = gpt2.to_single_token(answer)

# Getting model's prediction
top_prob, token_id_prediction = logits[0, -1].softmax(-1).max(-1)
top_prob_sae, token_id_prediction_sae = logits_sae[0, -1].softmax(-1).max(-1)

print(f"""Standard model:
    top prediction = {gpt2.to_string(token_id_prediction)!r}
    prob = {top_prob.item():.2%}
SAE reconstruction:
    top prediction = {gpt2.to_string(token_id_prediction_sae)!r}
    prob = {top_prob_sae.item():.2%}
""")

Tokenized prompt: ['<|endoftext|>Mitigating the risk of extinction from AI should be a global']
Tokenized answer: [' priority']


Performance on answer token:
Rank: 4206     Logit:  4.22 Prob:  0.00% Token: | priority|

Top 0th token. Logit: 12.07 Prob:  6.23% Token: |
|
Top 1th token. Logit: 11.57 Prob:  3.77% Token: |The|
Top 2th token. Logit: 11.12 Prob:  2.41% Token: |"|
Top 3th token. Logit: 10.90 Prob:  1.94% Token: |A|
Top 4th token. Logit: 10.85 Prob:  1.83% Token: |I|
Top 5th token. Logit: 10.39 Prob:  1.16% Token: |In|
Top 6th token. Logit: 10.36 Prob:  1.13% Token: |.|
Top 7th token. Logit: 10.15 Prob:  0.91% Token: |It|
Top 8th token. Logit: 10.10 Prob:  0.87% Token: |S|
Top 9th token. Logit:  9.99 Prob:  0.78% Token: |This|


Ranks of the answer tokens: [(' priority', 4206)]

Tokenized prompt: ['<|endoftext|>Mitigating the risk of extinction from AI should be a global']
Tokenized answer: [' priority']


...

Tokenized prompt: ['<|endoftext|>Mitigating the risk of extinction from AI should be a global']
Tokenized answer: [' priority']


Performance on answer token:
Rank: 4178     Logit:  4.16 Prob:  0.00% Token: | priority|

Top 0th token. Logit: 11.95 Prob:  5.76% Token: |
|
Top 1th token. Logit: 11.51 Prob:  3.68% Token: |The|
Top 2th token. Logit: 11.22 Prob:  2.76% Token: |"|
Top 3th token. Logit: 10.91 Prob:  2.03% Token: |I|
Top 4th token. Logit: 10.87 Prob:  1.94% Token: |A|
Top 5th token. Logit: 10.35 Prob:  1.16% Token: |In|
Top 6th token. Logit: 10.24 Prob:  1.04% Token: |.|
Top 7th token. Logit: 10.21 Prob:  1.01% Token: |S|
Top 8th token. Logit: 10.16 Prob:  0.96% Token: |It|
Top 9th token. Logit: 10.06 Prob:  0.87% Token: |This|


Ranks of the answer tokens: [(' priority', 4178)]

Standard model:
    top prediction = ' priority'
    prob = 52.99%
SAE reconstruction:
    top prediction = ' priority'
    prob = 39.84%



In [9]:
_, cache = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

for name, param in cache.items():
    if "hook_sae" in name:
        print(f"{name:<43}: {tuple(param.shape)}")

blocks.7.hook_resid_pre.hook_sae_input     : (1, 13, 768)
blocks.7.hook_resid_pre.hook_sae_acts_pre  : (1, 13, 24576)
blocks.7.hook_resid_pre.hook_sae_acts_post : (1, 13, 24576)
blocks.7.hook_resid_pre.hook_sae_recons    : (1, 13, 768)
blocks.7.hook_resid_pre.hook_sae_output    : (1, 13, 768)


In [10]:
# Get top activations on final token
_, cache = gpt2.run_with_cache_with_saes(
    prompt,
    saes=[gpt2_sae],
    stop_at_layer=_get_hook_layer(gpt2_sae) + 1,
)
sae_acts_post = cache[f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post"][0, -1, :]

# Plot line chart of latent activations
px.line(
    sae_acts_post.cpu().numpy(),
    title=f"Latent activations at the final token position ({sae_acts_post.nonzero().numel()} alive)",
    labels={"index": "Latent", "value": "Activation"},
    width=1000,
).update_layout(showlegend=False).show()

# Print the top 5 latents, and inspect their dashboards
for act, ind in zip(*sae_acts_post.topk(3)):
    print(f"Latent {ind} had activation {act:.2f}")
    display_dashboard(latent_idx=ind)

Latent 17676 had activation 48.18
https://neuronpedia.org/gpt2-small/7-res-jb/17676?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


Latent 17053 had activation 10.77
https://neuronpedia.org/gpt2-small/7-res-jb/17053?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


Latent 792 had activation 8.90
https://neuronpedia.org/gpt2-small/7-res-jb/792?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


In [11]:
logits_no_saes, cache_no_saes = gpt2.run_with_cache(prompt)

gpt2_sae.use_error_term = False
logits_with_sae_recon, cache_with_sae_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

gpt2_sae.use_error_term = True
logits_without_sae_recon, cache_without_sae_recon = gpt2.run_with_cache_with_saes(prompt, saes=[gpt2_sae])

# Both SAE caches contain the hook values
assert f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post" in cache_with_sae_recon
assert f"{gpt2_sae.cfg.metadata.hook_name}.hook_sae_acts_post" in cache_without_sae_recon

# But final output will be different, because we don't use SAE reconstructions when use_error_term
t.testing.assert_close(logits_no_saes, logits_without_sae_recon)
logit_diff_from_sae = (logits_no_saes - logits_with_sae_recon).abs().mean()
print(f"Average logit diff from using SAE reconstruction: {logit_diff_from_sae:.4f}")

Average logit diff from using SAE reconstruction: 0.4117


In [12]:
gpt2_act_store = ActivationsStore.from_sae(
    model=gpt2,
    sae=gpt2_sae,
    dataset="NeelNanda/pile-10k",
    streaming=True,
    store_batch_size_prompts=16,
    n_batches_in_buffer=32,
    device=str(device),
)

# Example of how you can use this:
tokens = gpt2_act_store.get_batch_tokens()
assert tokens.shape == (gpt2_act_store.store_batch_size_prompts, gpt2_act_store.context_size)

/Users/shubmeister/miniconda3/envs/arena/lib/python3.11/site-packages/sae_lens/training/activations_store.py:455: UserWarning: Dataset is not tokenized. Pre-tokenizing will improve performance and allows for more control over special tokens. See https://decoderesearch.github.io/SAELens/training_saes/#pretokenizing-datasets for more info.
  warnings.warn(
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3181 > 1024). Running this sequence through the model will result in indexing errors


In [13]:
display_dashboard(latent_idx=9)

https://neuronpedia.org/gpt2-small/7-res-jb/9?embed=true&embedexplanation=true&embedplots=true&embedtest=true&height=300


In [14]:
def show_activation_histogram(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 200,
):
    """
    Displays the activation histogram for a particular latent, computed across `total_batches`
    batches from `act_store`.
    """
    sae_acts_post_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"
    all_positive_acts = []

    for i in tqdm(range(total_batches), desc="Computing activations for histogram"):
        tokens = act_store.get_batch_tokens()
        _, cache = model.run_with_cache_with_saes(
            tokens,
            saes=[sae],
            stop_at_layer=_get_hook_layer(sae) + 1,
            names_filter=[sae_acts_post_hook_name],
        )
        acts = cache[sae_acts_post_hook_name][..., latent_idx]
        all_positive_acts.extend(acts[acts > 0].cpu().tolist())

    frac_active = len(all_positive_acts) / (total_batches * act_store.store_batch_size_prompts * act_store.context_size)

    px.histogram(
        all_positive_acts,
        nbins=50,
        title=f"ACTIVATIONS DENSITY {frac_active:.3%}",
        labels={"value": "Activation"},
        width=800,
        template="ggplot2",
        color_discrete_sequence=["darkorange"],
    ).update_layout(bargap=0.02, showlegend=False).show()

In [15]:
def get_k_largest_indices(x: Float[Tensor, "batch seq"], k: int, buffer: int = 0) -> Int[Tensor, "k 2"]:
    """
    The indices of the top k elements in the input tensor, i.e. output[i, :] is the (batch, seqpos)
    value of the i-th largest element in x.

    Won't choose any elements within `buffer` from the start or end of their sequence.
    """
    if buffer > 0:
        x = x[:, buffer:-buffer]
    indices = x.flatten().topk(k=k).indices
    rows = indices // x.size(1)
    cols = indices % x.size(1) + buffer
    return t.stack((rows, cols), dim=1)


x = t.arange(40, device=device).reshape((2, 20))
x[0, 10] += 50  # 2nd highest value
x[0, 11] += 100  # highest value
x[1, 1] += 150  # not inside buffer (it's less than 3 from the start of the sequence)
top_indices = get_k_largest_indices(x, k=2, buffer=3)
assert top_indices.tolist() == [[0, 11], [0, 10]]


def index_with_buffer(
    x: Float[Tensor, "batch seq"], indices: Int[Tensor, "k 2"], buffer: int | None = None
) -> Float[Tensor, " k *buffer_x2_plus1"]:
    """
    Indexes into `x` with `indices` (which should have come from the `get_k_largest_indices`
    function), and takes a +-buffer range around each indexed element. If `indices` are less than
    `buffer` away from the start of a sequence then we just take the first `2*buffer+1` elems (same
    for at the end of a sequence).

    If `buffer` is None, then we don't add any buffer and just return the elements at the given indices.
    """
    rows, cols = indices.unbind(dim=-1)
    if buffer is not None:
        rows = einops.repeat(rows, "k -> k buffer", buffer=buffer * 2 + 1)
        cols[cols < buffer] = buffer
        cols[cols > x.size(1) - buffer - 1] = x.size(1) - buffer - 1
        cols = einops.repeat(cols, "k -> k buffer", buffer=buffer * 2 + 1) + t.arange(
            -buffer, buffer + 1, device=cols.device
        )
    return x[rows, cols]


x_top_values_with_context = index_with_buffer(x, top_indices, buffer=3)
assert x_top_values_with_context[0].tolist() == [
    8,
    9,
    10 + 50,
    11 + 100,
    12,
    13,
    14,
]  # highest value in the middle
assert x_top_values_with_context[1].tolist() == [
    7,
    8,
    9,
    10 + 50,
    11 + 100,
    12,
    13,
]  # 2nd highest value in the middle


def display_top_seqs(data: list[tuple[float, list[str], int]]):
    """
    Given a list of (activation: float, str_toks: list[str], seq_pos: int), displays a table of
    these sequences, with the relevant token highlighted.

    We also turn newlines into "\\n", and remove unknown tokens � (usually weird quotation marks)
    for readability.
    """
    table = Table("Act", "Sequence", title="Max Activating Examples", show_lines=True)
    for act, str_toks, seq_pos in data:
        formatted_seq = (
            "".join([f"[b u green]{str_tok}[/]" if i == seq_pos else str_tok for i, str_tok in enumerate(str_toks)])
            .replace("�", "")
            .replace("\n", "↵")
        )
        table.add_row(f"{act:.3f}", repr(formatted_seq))
    rprint(table)


example_data = [
    (0.5, [" one", " two", " three"], 0),
    (1.5, [" one", " two", " three"], 1),
    (2.5, [" one", " two", " three"], 2),
]
display_top_seqs(example_data)

  Max Activating Examples   
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Act   ┃ Sequence         ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 0.500 │ ' one two three' │
├───────┼──────────────────┤
│ 1.500 │ ' one two three' │
├───────┼──────────────────┤
│ 2.500 │ ' one two three' │
└───────┴──────────────────┘

In [ ]:
def fetch_max_activating_examples(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 10,
    buffer: int = 10,
) -> list[tuple[float, list[str], int]]:
    """
    Returns the max activating examples across a number of batches from the activations store.
    """
    sae_acts_post_hook_name = f"{sae.cfg.metadata.hook_name}.hook_sae_acts_post"

    # Create list to store the top k activations for each batch. Once we're done,
    # we'll filter this to only contain the top k over all batches
    data = []

    for _ in tqdm(range(total_batches), desc="Computing activations for max activating examples"):
        tokens = act_store.get_batch_tokens()
        _, cache = model.run_with_cache_with_saes(
            tokens,
            saes=[sae],
            stop_at_layer=_get_hook_layer(sae) + 1,
            names_filter=[sae_acts_post_hook_name],
        )
        acts = cache[sae_acts_post_hook_name][..., latent_idx]

        # Get largest indices, get the corresponding max acts, and get the surrounding indices
        k_largest_indices = get_k_largest_indices(acts, k=k, buffer=buffer)
        tokens_with_buffer = index_with_buffer(tokens, k_largest_indices, buffer=buffer)
        str_toks = [model.to_str_tokens(toks) for toks in tokens_with_buffer]
        top_acts = index_with_buffer(acts, k_largest_indices).tolist()
        data.extend(list(zip(top_acts, str_toks, [buffer] * len(str_toks))))

    return sorted(data, key=lambda x: x[0], reverse=True)[:k]

buffer = 10
data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9, buffer=buffer, k=5)
display_top_seqs(data)

# Test one of the results, to see if it matches the expected output
first_seq_str_tokens = data[0][1]
assert first_seq_str_tokens[buffer] == " new"

In [17]:
data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=16873, total_batches=200)
display_top_seqs(data)

Computing activations for max activating examples:   0%|          | 0/200 [00:00<?, ?it/s]

                                           Max Activating Examples                                            
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Act   ┃ Sequence                                                                                           ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 5.057 │ " Oath, which involves vowing 'not to bring into the Library or kindle therein any fire or flame"  │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 4.720 │ ": 'cvrst be he yt moves my bones'.↵↵Anne Hathaway's Cottage"                                      │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 4.320 │ "'s Oath, which involves vowing 'not to bring into the Library or kindle therein any fire or"      │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 4.257 │ ", which involves vowing 'not to bring into the Library or kindle therein any fire or flame'."     │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.999 │ '15 July), it will rain for a further 40 days and 40 nights.↵↵Soggy ground'                        │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.822 │ '↵↵In AD 71 the Romans built a garrison called Eboracum, which in time became a'                   │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.753 │ ": 'A horse, a horse, my kingdom for a horse'. Well, actually, he didn't"                          │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.558 │ ' of the world as described in Genesis and the Book of Revelations.↵↵Undercroft, Treasury & Crypt' │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.433 │ ' elf maiden who gave up her immortality to be with him.↵↵2Activities↵↵A quint'                    │
├───────┼────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.403 │ " its ominous epitaph: 'cvrst be he yt moves my bones'.↵↵Anne Hath"                                │
└───────┴────────────────────────────────────────────────────────────────────────────────────────────────────┘

In [18]:
def get_k_largest_indices(
    x: Float[Tensor, "batch seq"],
    k: int,
    buffer: int = 0,
    no_overlap: bool = True,
) -> Int[Tensor, "k 2"]:
    """
    Returns the tensor of (batch, seqpos) indices for each of the top k elements in the tensor x.

    Args:
        buffer:     We won't choose any elements within `buffer` from the start or end of their seq
                    (this helps if we want more context around the chosen tokens).
        no_overlap: If True, this ensures that no 2 top-activating tokens are in the same seq and
                    within `buffer` of each other.
    """
    assert buffer * 2 < x.size(1), "Buffer is too large for the sequence length"
    assert not no_overlap or k <= x.size(0), "Not enough sequences to have a different token in each sequence"

    if buffer > 0:
        x = x[:, buffer:-buffer]

    indices = x.flatten().argsort(-1, descending=True)
    rows = indices // x.size(1)
    cols = indices % x.size(1) + buffer

    if no_overlap:
        unique_indices = t.empty((0, 2), device=x.device).long()
        while len(unique_indices) < k:
            unique_indices = t.cat((unique_indices, t.tensor([[rows[0], cols[0]]], device=x.device)))
            is_overlapping_mask = (rows == rows[0]) & ((cols - cols[0]).abs() <= buffer)
            rows = rows[~is_overlapping_mask]
            cols = cols[~is_overlapping_mask]
        return unique_indices

    return t.stack((rows, cols), dim=1)[:k]


x = t.arange(40, device=device).reshape((2, 20))
x[0, 10] += 150  # highest value
x[0, 11] += 100  # 2nd highest value, but won't be chosen because of overlap
x[1, 10] += 50  # 3rd highest, will be chosen
top_indices = get_k_largest_indices(x, k=2, buffer=3)
assert top_indices.tolist() == [[0, 10], [1, 10]]


data = fetch_max_activating_examples(gpt2, gpt2_sae, gpt2_act_store, latent_idx=16873, total_batches=200)
display_top_seqs(data)

Computing activations for max activating examples:   0%|          | 0/200 [00:00<?, ?it/s]

                                              Max Activating Examples                                              
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Act   ┃ Sequence                                                                                                ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 4.127 │ '<|endoftext|> created by bootfuls of earth, brought by nobles attending the coronations as an          │
│       │ acknowledgement of the'                                                                                 │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 4.016 │ ' or the head of John the Baptist to the body of Christ himself. The chapel is owned by the Episcopal'  │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.947 │ "bey's setting as 'everywhere peace, everywhere serenity, and a marvellous freedom from"                │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.861 │ ' a cloud/That floats on high over hills and dales/When all at once I saw a crowd'                      │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.275 │ " to pen the immortal lines: 'I wandered lonely as a cloud/That floats on high over hills and"          │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 3.034 │ "' and was linked to a prophecy that its death would mean curtains for the town. The tree died in"      │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 2.809 │ ' 15th-century wooden representation of the biblical figure of Jesse.↵↵Next door, the restored 13'      │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 2.787 │ " Ness beast. I am certain that this creature was of a prehistoric species.'↵↵The London newspapers     │
│       │ couldn"                                                                                                 │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 2.717 │ " men; and when Edward I's son was made by him the Prince of Wales in 1301, the"                        │
├───────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ 2.603 │ " the old lady's shawl.↵↵2Activities↵↵ oVale of Rheidol"                                                │
└───────┴─────────────────────────────────────────────────────────────────────────────────────────────────────────┘

In [23]:
def show_top_logits(
    model: HookedSAETransformer,
    sae: SAE,
    latent_idx: int,
    k: int = 10,
) -> None:
    """
    Displays the top & bottom logits for a particular latent.
    """
    logits = sae.W_dec[latent_idx] @ model.W_U

    pos_logits, pos_token_ids = logits.topk(k)
    pos_tokens = [model.to_string(t) for t in pos_token_ids]
    neg_logits, neg_token_ids = logits.topk(k, largest=False)
    neg_tokens = [model.to_string(t) for t in neg_token_ids]

    print(
        tabulate(
            zip(map(repr, neg_tokens), neg_logits, map(repr, pos_tokens), pos_logits),
            headers=["Bottom tokens", "Value", "Top tokens", "Value"],
            tablefmt="simple_outline",
            stralign="right",
            numalign="left",
            floatfmt="+.3f",
        )
    )
    
show_top_logits(gpt2, gpt2_sae, latent_idx=9)
tests.test_show_top_logits(show_top_logits, gpt2, gpt2_sae)

┌─────────────────┬─────────┬─────────────────┬─────────┐
│   Bottom tokens │ Value   │      Top tokens │ Value   │
├─────────────────┼─────────┼─────────────────┼─────────┤
│           'Zip' │ -0.774  │          'bies' │ +1.327  │
│       'acebook' │ -0.761  │           'bie' │ +1.297  │
│           'lua' │ -0.737  │     ' arrivals' │ +1.218  │
│        'ashtra' │ -0.728  │    ' additions' │ +1.018  │
│       'ONSORED' │ -0.708  │      ' edition' │ +0.994  │
│           'OGR' │ -0.705  │   ' millennium' │ +0.966  │
│      'umenthal' │ -0.703  │   ' generation' │ +0.962  │
│        'ecause' │ -0.697  │     ' entrants' │ +0.923  │
│          'icio' │ -0.692  │        ' hires' │ +0.919  │
│          'cius' │ -0.692  │ ' developments' │ +0.881  │
└─────────────────┴─────────┴─────────────────┴─────────┘
All tests in `test_show_top_logits` passed!


In [39]:
def get_autointerp_df(sae_release="gpt2-small-res-jb", sae_id="blocks.7.hook_resid_pre") -> pd.DataFrame:
    release = get_pretrained_saes_directory()[sae_release]
    neuronpedia_id = release.neuronpedia_id[sae_id]

    batch_idx = 7 // 512
    file_name = f"batch-{batch_idx}.jsonl.gz"
    
    url = f"https://neuronpedia-datasets.s3.amazonaws.com/v1/{neuronpedia_id}/explanations/{file_name}"
    return pd.read_json(url, lines=True)


explanations_df = get_autointerp_df()
explanations_df.head()

,id,modelId,layer,index,authorId,description,embedding,typeName,explanationModelName,umap_x,umap_y,umap_cluster,umap_log_feature_sparsity
0,clsxrpe1y00a3enuxmoxto0tx,gpt2-small,7-res-jb,0,clkht01d40000jv08hvalcvly,mathematical equations involving variables and...,"[0.007877588,0.034259304,-0.04119322,-0.044024...",oai_token-act-pair,gpt-3.5-turbo,7.606389,7.829482,0,-2.918194
1,ioiz5gmhiky2scql7dd35r34h,gpt2-small,7-res-jb,0,clkht01d40000jv08hvalcvly,mathematical functions and their relationship...,"[-0.08883373,0.032225072,-0.045294233,-0.03626...",oai_token-act-pair,gpt-4o-mini,0.000000,0.000000,0,0.000000
2,clsxrpf6600ajenux0q8smeor,gpt2-small,7-res-jb,1,clkht01d40000jv08hvalcvly,words related to law enforcement,"[-0.0074199713,0.007654639,-0.044788018,0.0595...",oai_token-act-pair,gpt-3.5-turbo,11.570288,5.130136,1,-3.382934
3,ifudwk8msryqzjgloa8qufh9s,gpt2-small,7-res-jb,1,clkht01d40000jv08hvalcvly,references to law enforcement and legal frame...,"[-0.040116165,-0.018509656,-0.05407583,-0.0099...",oai_token-act-pair,gpt-4o-mini,0.000000,0.000000,0,0.000000
4,clsxrpp1800fjenuxr1n7qkb6,gpt2-small,7-res-jb,10,clkht01d40000jv08hvalcvly,"phrases or sentences starting with ""We""","[-0.014056828,-0.009129634,-0.057800416,-0.064...",oai_token-act-pair,gpt-3.5-turbo,2.557040,5.533885,9,-2.903090


In [42]:
def create_prompt(
    model: HookedSAETransformer,
    sae: SAE,
    act_store: ActivationsStore,
    latent_idx: int,
    total_batches: int = 100,
    k: int = 15,
    buffer: int = 10,
) -> dict[Literal["system", "user", "assistant"], str]:
    """
    Returns the system, user & assistant prompts for autointerp.
    """
    data = fetch_max_activating_examples(model, sae, act_store, latent_idx, total_batches, k, buffer)
    str_formatted_examples = "\n".join(
        f"{i + 1}. {''.join(f'<<{tok}>>' if j == buffer else tok for j, tok in enumerate(seq[1]))}"
        for i, seq in enumerate(data)
    )

    return {
        "system": "We're studying neurons in a neural network. Each neuron activates on some particular word or concept in a short document. The activating words in each document are indicated with << ... >>. Look at the parts of the document the neuron activates for and summarize in a single sentence what the neuron is activating on. Try to be specific in your explanations, although don't be so specific that you exclude some of the examples from matching your explanation. Pay attention to things like the capitalization and punctuation of the activating words or concepts, if that seems relevant. Keep the explanation as short and simple as possible, limited to 20 words or less. Omit punctuation and formatting. You should avoid giving long lists of words.",
        "user": f"""The activating documents are given below:\n\n{str_formatted_examples}""",
        "assistant": "this neuron fires on",
    }


# Test your function
prompts = create_prompt(gpt2, gpt2_sae, gpt2_act_store, latent_idx=9, total_batches=100, k=15, buffer=8)
assert prompts["system"].startswith("We're studying neurons in a neural network.")
assert prompts["assistant"] == "this neuron fires on"

Computing activations for max activating examples:   0%|          | 0/100 [00:00<?, ?it/s]